In [1]:
import torch # type: ignore
import torchvision # type: ignore
import torch.nn as nn # type: ignore
import torch_directml # type: ignore

In [2]:
device = torch_directml.device(0)
print(torch_directml.device_name(0))

AMD Radeon RX 6800S 


In [3]:
model = torchvision.models.resnet101().to(device)
num_iter = 10
criterion = nn.CrossEntropyLoss(reduction='mean')
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)

In [6]:
batch_size = 100
optimizer.zero_grad()
for i in range(num_iter):
    inputs = torch.randn(batch_size, 3, 224, 224).to(device)
    labels = torch.LongTensor(batch_size).random_(0, 100).to(device)
    loss = criterion(model(inputs), labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print("Batch done")

RuntimeError: Could not allocate tensor with 60211200 bytes. There is not enough GPU video memory available!

In [4]:
needed_batch_size = 100
tolerated_batch_size = 5
accumulation_steps = needed_batch_size // tolerated_batch_size
optimizer.zero_grad()
for i in range(num_iter * accumulation_steps):
    inputs = torch.randn(tolerated_batch_size, 3, 224, 224).to(device)
    labels = torch.LongTensor(tolerated_batch_size).random_(0, 100).to(device)
    loss = criterion(model(inputs), labels)
    loss = loss / accumulation_steps
    loss.backward()
    if (i + 1) % accumulation_steps == 0:
        optimizer.step()
        optimizer.zero_grad()
        print("Batch done")

C:\Alexey\Projects\Udemy\NN_learning\.venv\Lib\site-packages\torch\optim\adam.py:534: UserWarning: The operator 'aten::lerp.Scalar_out' is not currently supported on the DML backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at C:\__w\1\s\pytorch-directml-plugin\torch_directml\csrc\dml\dml_cpu_fallback.cpp:17.)
  torch._foreach_lerp_(device_exp_avgs, device_grads, 1 - beta1)


Batch done
Batch done
Batch done
Batch done
Batch done
Batch done
Batch done
Batch done
Batch done
Batch done
